> In Google Colab, choose a GPU with bfloat16 support (L4 or A100) under Runtime, Change runtime type.

This section is adapted from Daniel Bourke's [VLM fine-tuning tutorial](https://github.com/mrdbourke/learn-huggingface/blob/main/notebooks/hugging_face_vlm_fine_tune_tutorial.ipynb), which the video below walks through.


## Task

<iframe width="100%" height="400" src="https://www.youtube.com/embed/_EMfJSmLSKE" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

You fine-tune a small vision language model (VLM) for structured data extraction. The model extracts food and drink items from an image and returns them as JSON.

The task is:

* Input: An image that may or may not contain food
* Output: Structured data that indicates whether the image contains food, together with food and drink tags

For example:

* Input (image): A photo of pork ribs, a tomato, grilled onions, carrots, and cilantro.
* Output:

```
{
    "is_food": 1,
    "image_title": "grilled ribs roasted carrots grilled onions tomato salsa",
    "food_items": [
        "rib meat",
        "onion",
        "grilled onion slice",
        "garlic",
        "rib bone",
        "cilantro",
        "cherry tomato",
        "carrot"
    ],
    "drink_items": []
}
```

## Vision Language Models

A Vision Language Model (VLM) combines visual and language processing in one model.

It is also called a multimodal language model (MLLM). Hugging Face lists these models under `Image-Text-to-Text`.

A VLM allows an LLM to process images and, in some cases, videos.

VLM applications include:

* Extracting information from documents.
* Identifying food and drinks in an image for nutrition tracking.
* Detecting potential visual defects in a manufacturing plant.
* Monitoring an area for security purposes.

## Small Vision Language Model fine-tuning

1. Local ownership: You can run the model on your own hardware and keep the data private.

2. Task scope: This task is simple enough for a small language model, which can run at lower cost.

3. Offline inference: The model can run without API calls and at lower cost.

4. Batch inference: Local batches can be faster than individual API calls. APIs can also process batches, but still incur usage.

5. Task-specific accuracy: The base model performed poorly on this task, but fine-tuning improved it.

> Traditional computer vision models such as ViT (Vision Transformer) work for many tasks. A VLM can handle different tasks when you specify the inputs and outputs.

## Fine-tuning as token prediction

VLMs and LLMs receive tokens as input and produce tokens as output.

A token is a numerical representation of data.

A model operates on numbers, not on raw images, text or video.

Each input must therefore be converted to numbers.

Here, data has a broad meaning.

It can include text, images, video, audio, DNA sequences, or spreadsheets.

A VLM or LLM receives an input token sequence and predicts the tokens that follow it.

You can therefore describe the task as **tokens in, tokens out**.

Choose the input tokens and the tokens that the model should return.

In this example, the input consists of an image and a string.

The output is structured information about food and drinks.

This use case is specific, but the same **tokens in, tokens out** formulation applies to many tasks.

Given a suitable dataset, you can fine-tune a VLM or LLM for the corresponding task.

## Import dependencies

In Google Colab, the first cell installs the packages that Colab does not include.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "ray[train]" "transformers>=5" "trl>=1.0" peft num2words


In [ ]:
import os

import torch
import transformers
import trl

# Training needs a GPU with bfloat16 support
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Using device: {device}")
print(f"[INFO] transformers {transformers.__version__}, trl {trl.__version__}")


## Settings

These settings control the size of the run and whether the results are uploaded. Set `NUM_TRAIN_SAMPLES` to a small number, such as 100, for a quick run that checks the code end to end. Set `PUSH_TO_HUB` to `True` to upload the fine-tuned model and the demo to your Hugging Face account.

In [ ]:
NUM_TRAIN_SAMPLES = None  # None trains on the whole training split
NUM_EPOCHS = 4
PUSH_TO_HUB = os.environ.get("PUSH_TO_HUB", "0") == "1"
HF_ACCOUNT = os.environ.get("HF_ACCOUNT")  # None uploads to your own Hugging Face username


## Load the dataset

A VLM dataset consists of image and text pairs.

For each input image, specify the text that the model should produce.

This dataset contains:

* 1000 food images (from Food101)
* 500 not food images (randomly sampled from the internet)

The images were labeled for food and drink extraction with the larger [`Qwen3-VL-8B-Instruct`](https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct) model. They were then converted into a [Hugging Face Image Dataset](https://huggingface.co/docs/datasets/en/image_dataset#imagefolder).

> The [YouTube livestream](https://www.youtube.com/live/X-nYCCviCyI?si=kPKhTxXbc7pUztAC) shows how the dataset was built and uploaded to Hugging Face.

In [ ]:
from datasets import load_dataset
from PIL import Image

dataset = load_dataset("mrdbourke/FoodExtract-1k-Vision")
dataset

In [ ]:
example_sample = dataset["train"][0]

example_sample

The dataset contains two fields used in this section:

* `image`: The input image.
* `output_label_json`: The structured output that the fine-tuned model should return. The model returns text, so you train it to produce valid JSON that can be parsed and stored in a database.

In [ ]:
example_input = example_sample["image"]
example_output = example_sample["output_label_json"]

print(f"[INFO] Example desired output:\n{example_output}")

print(f"[INFO] Example image input:")
example_input

## Format the data for a VLM

The VLM expects data formatted as a sequence of chat messages.

The system prompt is:

```
SYSTEM_MESSAGE = """You are an expert food and drink image extractor.
You provide structured data to visual inputs classifying them as edible food/drink or not.
As well as titling the image with a simple food/drink related caption.
Finally you extract any and all visible food/drink items to lists.
"""
```

The user prompt is:

````
USER_PROMPT = """Classify the given input image into food or not and if edible food or drink items are present, extract those to a list. If no food/drink items are visible, return empty lists.

Only return valid JSON in the following form:

```json
{
  'is_food': 0, # int - 0 or 1 based on whether food/drinks are present (0 = no foods visible, 1 = foods visible)
  'image_title': '', # str - short food-related title for what foods/drinks are visible in the image, leave blank if no foods present
  'food_items': [], # list[str] - list of visible edible food item nouns
  'drink_items': [] # list[str] - list of visible edible drink item nouns
}
```
"""
````

First, evaluate the base model with these prompts.

In [ ]:
# System message
SYSTEM_MESSAGE = """You are an expert food and drink image extractor.
You provide structured data to visual inputs classifying them as edible food/drink or not.
As well as titling the image with a simple food/drink related caption.
Finally you extract any and all visible food/drink items to lists.
"""

# User prompt with image input as well as desired output
USER_PROMPT = """Classify the given input image into food or not and if edible food or drink items are present, extract those to a list. If no food/drink items are visible, return empty lists.

Only return valid JSON in the following form:

```json
{
  'is_food': 0, # int - 0 or 1 based on whether food/drinks are present (0 = no foods visible, 1 = foods visible)
  'image_title': '', # str - short food-related title for what foods/drinks are visible in the image, leave blank if no foods present
  'food_items': [], # list[str] - list of visible edible food item nouns
  'drink_items': [] # list[str] - list of visible edible drink item nouns
}
```
"""

# Create helper function to map our input/output samples to conversational format
def format_data(sample):
    return {
        "messages": [

            # Message 0 - [SYSTEM] System Prompt (setting the scene)
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_MESSAGE}]
            },

            # Message 1 - [USER] User input (image + prompt pair)
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": sample["image"],
                    },
                    {
                        "type": "text",
                        "text": USER_PROMPT # Note: In a future extension, you might train the model to not require any text input and just go straight from image -> text output
                    }
                ],
            },

            # Message 2 - [MODEL] Ideal model output (e.g. our structured data format)
            {
                "role": "assistant",
                "content": [{"type": "text", "text": sample["output_label_json"]}]
            }
        ]
    }

formatted_sample = format_data(sample=example_sample)
formatted_sample

In [ ]:
# Format each sample as a list of chat messages, keeping the PIL image
dataset_processed = [format_data(sample) for sample in dataset["train"]]
dataset_processed[0]

### Training and validation split

Split the data into training and validation sets.

Fine-tune the model on the training set, then evaluate it on the validation set.

In [ ]:
import random
from pprint import pprint

def split_dataset(samples, seed=42, train_fraction=0.8):
    """Shuffle with a fixed seed and split 80% for training, 20% for validation."""
    rng = random.Random(seed)
    shuffled = rng.sample(samples, k=len(samples))
    n_train = round(train_fraction * len(shuffled))
    return shuffled[:n_train], shuffled[n_train:]

train_dataset, val_dataset = split_dataset(dataset_processed)
print(f"[INFO] Number of total samples: {len(dataset_processed)}")
print(f"[INFO] Number of train samples: {len(train_dataset)}")
print(f"[INFO] Number of eval samples: {len(val_dataset)}")

print(f"\n[INFO] Example training sample:\n")
pprint(train_dataset[0])
print(f"\n[INFO] Example validation sample:\n")
pprint(val_dataset[0])

## Base-model inference

Evaluate a base model on the task before fine-tuning.

* Candidate models: SmolVLM2-256M and SmolVLM2-500M
    * Model collection: https://huggingface.co/collections/HuggingFaceTB/smolvlm2-smallest-video-lm-ever
* An earlier experiment used [gemma-3n-E2B-it](https://huggingface.co/google/gemma-3n-E2B-it), but its baseline was already high, so there was little for fine-tuning to improve.

In [ ]:
# Get a base model input example
# We only want our model to see the user system/user inputs (not the desired response) so we index on the correct amount of messages
example_base_model_input = dataset_processed[0]["messages"][:2]
example_base_model_input

### Load a base model

Many VLMs are available for this task.

This project uses a small model with fewer than 1B parameters.

In earlier experiments, `google/gemma-3n-e2b-it` (5B parameters) performed well on this task.

This section uses [`HuggingFaceTB/SmolVLM2-500M-Video-Instruct`](https://huggingface.co/HuggingFaceTB/SmolVLM2-500M-Video-Instruct), which is 10 times smaller than `google/gemma-3n-e2b-it`.

The goal is to make this smaller model perform as well as the larger model.

> The Hugging Face Models page lists base models under the `Image-Text-to-Text` filter: https://huggingface.co/models?pipeline_tag=image-text-to-text

In [ ]:
from transformers import pipeline

# MODEL_ID = "google/gemma-3n-e2b-it"
# MODEL_ID = "HuggingFaceTB/SmolVLM2-256M-Video-Instruct"

MODEL_ID = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"

pipe = pipeline(
    "image-text-to-text",
    model=MODEL_ID,
    device="cuda",
    dtype=torch.bfloat16
)

In [ ]:
# Test the base model on an example input image
base_model_output = pipe(text=example_base_model_input,
                         max_new_tokens=256)

print(f'[INFO] Base model input (text only):\n{example_base_model_input}\n')
print(f'[INFO] Base model output:\n{base_model_output[0]["generated_text"][-1]["content"]}\n')
print(f'[INFO] Desired output:\n{dataset_processed[0]["messages"][-1]["content"][0]["text"]}')

The base model returns an empty list, although the image contains food items. Inspect the image to verify this.

In [ ]:
# Check example input image
example_base_model_input[1]["content"][0]["image"]

The image contains food.

The expected output has the following form:

```
{'is_food': 1,
 'image_title':
 'cheese plate',
 'food_items': ['toast', 'herb garnish', 'dipping sauce', 'cheese wedge'],
 'drink_items': []}
```

The base model does not produce this output. The rest of this section fine-tunes it for the task.

## Load the model directly

`transformers.pipeline` provides a simple interface for loading a model and running inference.

Direct loading gives access to the model components needed below.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

print(f"[INFO] Using model: {MODEL_ID}")

# Define model init args
model_kwargs = dict(
    attn_implementation="eager", # Note: could also use flash_attention_2 here for faster models, requires `pip install flash-attn` (see: https://github.com/Dao-AILab/flash-attention)
    dtype=torch.bfloat16,
    device_map="auto"
)

# Load the model and tokenizer/processor
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID,
                                                    **model_kwargs)
processor = AutoProcessor.from_pretrained(MODEL_ID)

Inspect the model architecture before modifying it.

In [ ]:
# Inspect our model's architecture
model

The model has four main components:

* `vision_model`: Encodes the image. SmolVLM2 uses [SigLIP](https://huggingface.co/google/siglip-base-patch16-512) as its vision encoder.
* `text_model`: The language model, [SmolLM2-360M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct). It processes the text tokens and the visual tokens as one sequence.
* `connector`: Turns the output of `vision_model` into visual tokens that `text_model` can read. It first merges each 4 × 4 block of neighboring patch features into one vector of 16 × 768 = 12,288 values (pixel shuffle), which cuts the number of visual tokens by a factor of 16. A linear layer then projects each vector to 960 values, the embedding size of `text_model`.
* `lm_head`: Produces tokens conditioned on the visual and language data.

Stage 1 of fine-tuning updates every component except `vision_model`, as in [SmolDocling](https://arxiv.org/abs/2503.11576).

Run one sample through `model`.

In [ ]:
# Apply the chat template
example_chat_template_input = processor.apply_chat_template(
    [example_base_model_input[1]], # Note: if we are running this way, format the system_prompt to either be removed or folded into the user_prompt
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(device)

# Measure how long our input prompt is (our model will generate tokens *after* the input prompt)
input_len = example_chat_template_input["input_ids"].shape[-1]
print(f"[INFO] Input length: {input_len}")

# Perform inference with the base model
with torch.no_grad():
    generation_output_full = model.generate(**example_chat_template_input,
                                            max_new_tokens=256,
                                            do_sample=False)
    generation_output_short = generation_output_full[0][input_len:]

# Decode the outputs from tokens -> readable text
decoded_outputs = processor.decode(generation_output_short,
                                   skip_special_tokens=True)

# Print inputs and outputs
print("\n"+"-"*50+"\n")
print(f"[INFO] This is the raw token (numerical) input to our model:\n")
print(example_chat_template_input["input_ids"][0])
print("\n"+"-"*50+"\n")
print(f"[INFO] This is the raw pixel values (numerical) input to our model:\n")
print(example_chat_template_input["pixel_values"][0])
print("\n"+"-"*50+"\n")
print(f"[INFO] This is the input to our model in human-readable form (note: our model will not see in the input like this):\n")
print(processor.decode(example_chat_template_input["input_ids"][0]))
print("\n"+"-"*50+"\n")
print(f"[INFO] Outputs from the model:\n")
print(decoded_outputs)

The base `model` does not return the requested structure either, just like the earlier `pipe`.

> The `<image>` special tokens represent the image in the text input. Each one is a placeholder. Its feature values come from `vision_model` through the `example_chat_template_input["pixel_values"]` key.

## Prepare model for training

Following [section 5.1 of the SmolDocling paper](https://arxiv.org/pdf/2503.11576), freeze `vision_model` and train the remaining LLM components.

With a larger dataset, training could use two stages. This dataset has about 1,000 training samples.

1. Train with a frozen vision encoder so that the LLM components learn the target output format.
2. Train the full model so that all features adapt to the target dataset.

This section trains stage 1 first. Stage 2 follows the comparison of the base and fine-tuned models.

Count the total and trainable parameters before and after freezing the vision encoder.

In [ ]:
def freeze_vision_encoder(model):
    """Turn off gradients in the vision encoder so only the language side trains."""
    for param in model.model.vision_model.parameters():
        param.requires_grad = False

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable
    return total, trainable, frozen

# Before freezing parameters
total_params, trainable_params, frozen_params = count_parameters(model)
print(f"[INFO] Params Total (before freezing the vision model): {total_params:,} | Params Trainable: {trainable_params:,} | Params Frozen: {frozen_params:,}")

# Freeze the vision encoder
freeze_vision_encoder(model)

# After freezing parameters
total_params, trainable_params, frozen_params = count_parameters(model)
print(f"[INFO] Params Total (after freezing the vision model): {total_params:,} | Params Trainable: {trainable_params:,} | Params Frozen: {frozen_params:,}")

## Create a data collator

A data collator groups image and text samples into batches.

It tells `transformers.Trainer` how to combine multiple samples for batch training.

Define two functions:

1. `convert_message_to_list_of_images`: Extracts images from a list of messages.
2. `make_collate_fn`: Returns a `collate_fn` that formats a group of dataset rows as chat messages and turns them into a training batch with a given processor.

In [ ]:
def convert_message_to_list_of_images(messages: list[dict]) -> list[Image.Image]:
    """Helper function to turn our message samples into lists of images.

    Note: this comes from here: https://ai.google.dev/gemma/docs/core/huggingface_vision_finetune_qlora"""
    image_inputs = []

    # Iterate through each conversation
    for msg in messages:
        # Get content (ensure it's a list)
        content = msg.get("content", [])
        if not isinstance(content, list):
            content = [content]

        # Check each content element for images
        for element in content:
            if isinstance(element, dict) and (
                "image" in element or element.get("type") == "image"
            ):
                # Get the image and convert to RGB
                if "image" in element:
                    image = element["image"]
                else:
                    image = element
                image_inputs.append(image.convert("RGB"))

    return image_inputs

def make_collate_fn(processor):
    """Return a collate function that formats dataset rows as chat messages and batches them with `processor`."""

    def collate_fn(examples):

        # Create empty lists for batches of text and images
        texts = []
        images = []

        # Loop through examples and preprocess them
        for example in examples:
            messages = format_data(example)["messages"]  # dataset row -> chat messages
            image_inputs = convert_message_to_list_of_images(messages)
            text = processor.apply_chat_template(
                messages, add_generation_prompt=False, tokenize=False
            )
            texts.append(text.strip())
            images.append(image_inputs)

        # Tokenize the texts and process the images
        batch_input = processor(text=texts,
                                images=images,
                                return_tensors="pt",
                                padding=True)

        # The labels are the input_ids and we mask the padding tokens and image tokens in the loss computation
        labels = batch_input["input_ids"].clone()

        # Get the image token
        image_token_id = processor.tokenizer.convert_tokens_to_ids("<image>")

        # Mask tokens for not being used in the loss computation
        labels[labels == processor.tokenizer.pad_token_id] = -100 # pad_token_id = 2
        labels[labels == image_token_id] = -100 # image_token_id = 49190, this is the `<image>` token which is a placeholder, we don't need our model to learn to predict those

        batch_input["labels"] = labels

        return batch_input

    return collate_fn

collate_fn = make_collate_fn(processor)

# For SmolVLM2, check the image token
image_token_id = processor.tokenizer.convert_tokens_to_ids("<image>")
print(f"[INFO] SmolVLM2 uses the following for the image_token_id: {image_token_id}, we mask this token as it is only a placeholder in our sequence of tokens (we don't need the model to learn to predict it).")

> During fine-tuning, use the tokenizer and special tokens defined for the selected model. The wrong tokenizer or tokens can produce invalid model inputs.

## Configure SFT

Supervised Fine-Tuning (SFT) trains the model on input and output pairs.

For each pair, the model learns to predict the output from the input.

SFT uses two classes:

1. `SFTConfig` (Supervised Fine-Tuning Config): Defines the training hyperparameters. See https://huggingface.co/docs/trl/en/sft_trainer#trl.SFTConfig
2. `SFTTrainer` (Supervised Fine-Tuning Trainer): Defines the model and data used for training. See https://huggingface.co/docs/trl/en/sft_trainer#trl.SFTTrainer


In [ ]:
from trl import SFTConfig

# Note: Many of these hyperparameters have been taken from the SmolDocling paper: https://arxiv.org/abs/2503.11576
training_args = SFTConfig(
    output_dir=os.path.abspath("smolvlm2-500m-FoodExtract-Vision-v1"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=4, # adjust based on the size of your GPU
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    bf16=True,
    save_total_limit=1,
    max_grad_norm=1.0,
    warmup_steps=0.03, # a value below 1 is a fraction of the total steps
    lr_scheduler_type="constant",
    push_to_hub=False,
    report_to="none",
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
    load_best_model_at_end=True,
    remove_unused_columns=False,
    disable_tqdm=True, # print the logged metrics instead of a progress bar
)

## Train on a Ray Train worker

The training loop runs inside a Ray Train worker instead of in the main Python process. A worker is a separate process that owns one GPU. It loads its own copy of the model and the data, runs `SFTTrainer`, and reports its metrics and checkpoints back to the driver, which is the process running this page.

`ScalingConfig` sets how many workers to start and whether each one gets a GPU. With one worker, as here, the run is the same as calling `trainer.train()` directly. With `num_workers=4` on a machine or cluster with four GPUs, Ray starts four workers, each worker trains on a quarter of every batch, and the gradients are averaged across them (data parallelism). The training function does not change.

The `freeze_vision` entry of the config decides whether the worker freezes the vision encoder, so stage 2 reuses the same function.

`ray.init()` starts Ray on the current machine. In Google Colab that machine has one GPU, so `num_workers` must stay at 1.

In [ ]:
# Free the GPU memory held by the models loaded above: the worker loads its own copy
del model, pipe
torch.cuda.empty_cache()


In [ ]:
def train_func(config):
    """Runs on each Ray Train worker: build the model and data, then train."""
    import json

    import ray.train
    import torch
    from datasets import load_dataset
    from ray.train.huggingface.transformers import RayTrainReportCallback, prepare_trainer
    from transformers import AutoModelForImageTextToText, AutoProcessor
    from trl import SFTTrainer

    # Each worker loads its own model and processor; Ray assigns the GPU
    model = AutoModelForImageTextToText.from_pretrained(
        config["model_id"], attn_implementation="eager", dtype=torch.bfloat16
    )
    processor = AutoProcessor.from_pretrained(config["model_id"])
    if config["freeze_vision"]:
        freeze_vision_encoder(model)

    # Rebuild the same train/validation split as the driver, as rows of the original dataset.
    # The collator formats each row into chat messages when it is batched.
    rows = load_dataset(config["dataset_id"])["train"]
    train_idx, val_idx = split_dataset(list(range(len(rows))))
    if config["num_train_samples"] is not None:
        train_idx = train_idx[: config["num_train_samples"]]
        val_idx = val_idx[: max(1, config["num_train_samples"] // 4)]
    train_split, val_split = rows.select(train_idx), rows.select(val_idx)

    trainer = SFTTrainer(
        model=model,
        args=config["training_args"],
        train_dataset=train_split,
        eval_dataset=val_split,
        processing_class=processor,
        data_collator=make_collate_fn(processor),
        callbacks=[RayTrainReportCallback()],  # report metrics and checkpoints to Ray
    )
    trainer = prepare_trainer(trainer)
    trainer.train()

    # The first worker saves the best model and the loss history for the driver
    if ray.train.get_context().get_world_rank() == 0:
        out = config["training_args"].output_dir
        trainer.save_model(out)
        processor.save_pretrained(out)
        with open(os.path.join(out, "log_history.json"), "w") as f:
            json.dump(trainer.state.log_history, f)


In [ ]:
import ray
from ray.train import CheckpointConfig, RunConfig, ScalingConfig
from ray.train.torch import TorchTrainer

ray.init(ignore_reinit_error=True, log_to_driver=False)  # keep the worker logs out of the page

ray_trainer = TorchTrainer(
    train_func,
    train_loop_config={
        "model_id": MODEL_ID,
        "dataset_id": "mrdbourke/FoodExtract-1k-Vision",
        "training_args": training_args,
        "num_train_samples": NUM_TRAIN_SAMPLES,
        "freeze_vision": True,
    },
    scaling_config=ScalingConfig(num_workers=1, use_gpu=True),
    run_config=RunConfig(
        name="smolvlm2-foodextract-sft",
        storage_path=os.path.abspath("ray_results"),
        checkpoint_config=CheckpointConfig(num_to_keep=1),
    ),
)
result = ray_trainer.fit()

import json
import math

def print_loss_table(output_dir):
    """Print the mean training loss and the validation metrics of each epoch."""
    # The worker saved its log history next to the model
    with open(os.path.join(output_dir, "log_history.json")) as f:
        history = json.load(f)
    # Training loss is logged every 25 steps, so average the values logged inside each epoch.
    # A value logged at epoch 1.32 belongs to epoch 2, which is why the epoch is rounded up.
    train_losses = {}
    for r in history:
        if "loss" in r:
            train_losses.setdefault(math.ceil(r["epoch"] - 1e-9), []).append(r["loss"])
    train_rows = {e: sum(v) / len(v) for e, v in train_losses.items()}
    print("epoch  mean train loss  val loss  val token accuracy")
    for r in history:
        if "eval_loss" in r:
            e = round(r["epoch"])
            print(f"{e:>5}  {train_rows.get(e, float('nan')):>15.4f}  {r['eval_loss']:>8.4f}  {r['eval_mean_token_accuracy']:>18.4f}")

print_loss_table(training_args.output_dir)
print(f"[INFO] Ray checkpoint: {result.checkpoint}")

ray.shutdown()  # stop the local Ray instance; the trained model is on disk


Training ran for 4 epochs over the 1,208 training images, 304 optimizer steps of 16 images each, in about 40 minutes on one GPU. The mean training loss fell in every epoch, from 0.191 in epoch 1 to 0.025 in epoch 4. The first epoch's mean is high because it includes the early steps, before the model had learned the output format. The validation loss was lowest after epoch 2 (0.075) and rose to 0.080 by epoch 4, so the last two epochs fit the training images more closely without helping on new ones. Because `load_best_model_at_end` is set, the saved model is the epoch 2 checkpoint, not the last one.

Token accuracy on the validation images stays near 98% in every epoch. Much of each answer is the same JSON structure and field names, which the model learns early, so this number says little about whether the food items are right. The comparison below looks at the items themselves.

### Training and validation loss curves

In [ ]:
import json

import matplotlib.pyplot as plt

# Read the log history the worker saved next to the model
with open(os.path.join(training_args.output_dir, "log_history.json")) as f:
    log_history = json.load(f)

# Extract training / validation loss
train_losses = [log["loss"] for log in log_history if "loss" in log]
epoch_train = [log["epoch"] for log in log_history if "loss" in log]
eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]
epoch_eval = [log["epoch"] for log in log_history if "eval_loss" in log]

# Plot the training loss
plt.plot(epoch_train, train_losses, label="Training Loss")
plt.plot(epoch_eval, eval_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()

## Upload the fine-tuned model to the Hugging Face Hub

This step runs only when `PUSH_TO_HUB` is `True`.

> Authenticate with the Hugging Face Hub through the [`huggingface_hub` package](https://huggingface.co/docs/huggingface_hub/en/guides/cli) before uploading.
>
> The [`hf auth login` documentation](https://huggingface.co/docs/huggingface_hub/en/guides/cli#hf-auth-login) explains how to link your computer to your Hugging Face profile with a [User Access Token](https://huggingface.co/docs/hub/security-tokens).

In [ ]:
import glob
import shutil

# Remove the intermediate checkpoints: only the final model files are uploaded
for checkpoint_dir in glob.glob(os.path.join(training_args.output_dir, "checkpoint-*")):
    shutil.rmtree(checkpoint_dir)
print(sorted(os.listdir(training_args.output_dir)))


In [ ]:
from huggingface_hub import HfApi

if PUSH_TO_HUB:
    api = HfApi()
    account = HF_ACCOUNT or api.whoami()["name"]
    repo_id = f"{account}/FoodExtract-Vision-SmolVLM2-500M-fine-tune-v1"

    api.create_repo(repo_id, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=training_args.output_dir, repo_id=repo_id, repo_type="model")
    print(f"[INFO] Uploaded the fine-tuned model to https://huggingface.co/{repo_id}")
else:
    print("[INFO] PUSH_TO_HUB is False, so the model was not uploaded")


## Base and fine-tuned model comparison

In [ ]:
CHECKPOINT_DIR_NAME = training_args.output_dir
print(f"[INFO] Using fine-tuned checkpoint: {CHECKPOINT_DIR_NAME}")
print(f"[INFO] Using base model checkpoint: {MODEL_ID}")

In [ ]:
from transformers import pipeline

# Load the base model (pt_pipe stands for "pretrained pipeline", 'pretrained model' is another word for 'base model'
pt_pipe = pipeline(
    "image-text-to-text",
    model=MODEL_ID,
    dtype=torch.bfloat16
)

# `do_sample = False` for deterministic responses
pt_pipe.model.generation_config.do_sample = False
pt_pipe.model.generation_config.pad_token_id = processor.tokenizer.eos_token_id

# Load the fine-tuned model
ft_pipe = pipeline(
    "image-text-to-text",
    model=CHECKPOINT_DIR_NAME,
    dtype=torch.bfloat16
)

# `do_sample = False` for deterministic responses
ft_pipe.model.generation_config.do_sample = False
ft_pipe.model.generation_config.pad_token_id = processor.tokenizer.eos_token_id

In [ ]:
pt_pipe.device, ft_pipe.device

In [ ]:
import random

# Pick a random sample from the validation dataset (fixed seed, so the page is reproducible)
random_val_sample = random.Random(0).choice(val_dataset)
random_val_sample_model_input = random_val_sample["messages"][1]
random_val_sample_image = random_val_sample_model_input["content"][0]["image"]
random_val_sample_model_output = random_val_sample["messages"][-1]

# Print out the ideal input and outputs
print(f"[INFO] Example model input:\n{random_val_sample_model_input}")
print()
_ideal = random_val_sample_model_output["content"][0]["text"]
print(f"[INFO] Example model ideal output:\n{_ideal}")

# Perform a forward pass with the base model
pt_pipe_output = pt_pipe(text=[random_val_sample_model_input],
                         max_new_tokens=256)
pt_pipe_output_text = pt_pipe_output[0]["generated_text"][-1]["content"]
print(f"\n[INFO] Generated output text from pre-trained model:\n{pt_pipe_output_text}\n")

# Perform a forward pass with the fine-tuned model
ft_pipe_output = ft_pipe(text=[random_val_sample_model_input],
                         max_new_tokens=256)
ft_pipe_output_text = ft_pipe_output[0]["generated_text"][-1]["content"]
print(f"\n[INFO] Generated output text from fine-tuned model:\n{ft_pipe_output_text}\n")

print(f"[INFO] Input image:")
random_val_sample_image.resize(size=(500, 500))

On this validation image of pan-fried dumplings, the base model again returns an empty list. The fine-tuned model returns the requested structure, marks the image as food and titles it "fried dumplings". Its item list is partly right. It finds the dumpling wrapper and the filling (as "dumpling meat"), turns the lettuce leaf of the label into a "dumpling leaf", and adds two sauce items that the label does not list.

Fine-tuning taught the model the output format reliably. The content of the lists is where it still makes mistakes, which is what the further experiments at the end of this section address.

## Stage 2: unfreeze the vision encoder

In stage 1 the vision encoder kept its pretrained weights, and only the language side learned the output format. Stage 2 is the second step of the SmolDocling recipe: it continues from the stage 1 model and trains every parameter, including the vision encoder, so that the image features can also adapt to food photos.

The vision encoder was pretrained on far more images than the 1,208 in the training split. A learning rate as large as the one of stage 1 could overwrite features that the language side has just learned to read. Stage 2 therefore uses a learning rate ten times smaller, `2e-5`, for two epochs. It evaluates the model once before the first step (`eval_on_start`), so the first row of its loss table, epoch 0, is the stage 1 model it starts from.

Unfreezing adds the parameters of the vision encoder to the trainable set.

In [ ]:
total_params = sum(p.numel() for p in ft_pipe.model.parameters())
vision_params = sum(p.numel() for p in ft_pipe.model.model.vision_model.parameters())
print(f"[INFO] Params Total: {total_params:,}")
print(f"[INFO] Trainable in stage 1 (vision encoder frozen): {total_params - vision_params:,}")
print(f"[INFO] Trainable in stage 2 (all parameters): {total_params:,} ({vision_params:,} of them in the vision encoder)")

In [ ]:
import dataclasses

NUM_EPOCHS_STAGE_2 = 2

# Same settings as stage 1, apart from the output folder, the learning rate and the number of epochs
training_args_stage_2 = dataclasses.replace(
    training_args,
    output_dir=os.path.abspath("smolvlm2-500m-FoodExtract-Vision-stage-2-v1"),
    num_train_epochs=NUM_EPOCHS_STAGE_2,
    learning_rate=2e-5,
    eval_on_start=True,  # evaluate the stage 1 model before the first step
)
print(f"[INFO] Stage 2 starts from: {CHECKPOINT_DIR_NAME}")
print(f"[INFO] Stage 2 saves to: {training_args_stage_2.output_dir}")

In [ ]:
ray.init(ignore_reinit_error=True, log_to_driver=False)

ray_trainer_stage_2 = TorchTrainer(
    train_func,
    train_loop_config={
        "model_id": CHECKPOINT_DIR_NAME,  # the stage 1 model
        "dataset_id": "mrdbourke/FoodExtract-1k-Vision",
        "training_args": training_args_stage_2,
        "num_train_samples": NUM_TRAIN_SAMPLES,
        "freeze_vision": False,
    },
    scaling_config=ScalingConfig(num_workers=1, use_gpu=True),
    run_config=RunConfig(
        name="smolvlm2-foodextract-sft-stage-2",
        storage_path=os.path.abspath("ray_results"),
        checkpoint_config=CheckpointConfig(num_to_keep=1),
    ),
)
result_stage_2 = ray_trainer_stage_2.fit()
print_loss_table(training_args_stage_2.output_dir)

ray.shutdown()

STAGE2_RESULTS_PLACEHOLDER

In [ ]:
CHECKPOINT_DIR_NAME_STAGE_2 = training_args_stage_2.output_dir

ft_pipe_stage_2 = pipeline(
    "image-text-to-text",
    model=CHECKPOINT_DIR_NAME_STAGE_2,
    dtype=torch.bfloat16
)
ft_pipe_stage_2.model.generation_config.do_sample = False
ft_pipe_stage_2.model.generation_config.pad_token_id = processor.tokenizer.eos_token_id

# The same validation image as in the comparison above
ft_pipe_stage_2_output = ft_pipe_stage_2(text=[random_val_sample_model_input],
                                         max_new_tokens=256)
ft_pipe_stage_2_output_text = ft_pipe_stage_2_output[0]["generated_text"][-1]["content"]
print(f"[INFO] Ideal output:\n{_ideal}\n")
print(f"[INFO] Stage 1 model:\n{ft_pipe_output_text}\n")
print(f"[INFO] Stage 2 model:\n{ft_pipe_stage_2_output_text}")

STAGE2_COMPARE_PLACEHOLDER

## Create a demo with Gradio

The demo requires three files:

* `app.py`: The application entry point for Hugging Face Spaces.
* `requirements.txt`: The dependencies that Hugging Face Spaces installs at startup.
* `README.md`: A description of the demo.

The demo also shows three example images from the validation split, saved to an `examples` folder.

In [ ]:
# Make a directory for the demo and its example images
os.makedirs("demos/FoodExtract-Vision-v1/examples", exist_ok=True)


In [ ]:
%%writefile demos/FoodExtract-Vision-v1/app.py
import torch
import gradio as gr

import spaces
from transformers import pipeline

BASE_MODEL_ID = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
FINE_TUNED_MODEL_ID = "aegean-ai/FoodExtract-Vision-SmolVLM2-500M-fine-tune-v1"
OUTPUT_TOKENS = 256

# Load original base model (no fine-tuning)
print(f"[INFO] Loading Original Model")
original_pipeline = pipeline(
    "image-text-to-text",
    model=BASE_MODEL_ID,

    dtype=torch.bfloat16,
    device_map="auto"
)

# Load fine-tuned model
print(f"[INFO] Loading Fine-tuned Model")
ft_pipe = pipeline(
    "image-text-to-text",
    model=FINE_TUNED_MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto"
)

def create_message(input_image):
    return [{'role': 'user',
 'content': [{'type': 'image',
   'image': input_image},
  {'type': 'text',
   'text': "Classify the given input image into food or not and if edible food or drink items are present, extract those to a list. If no food/drink items are visible, return empty lists.\n\nOnly return valid JSON in the following form:\n\n```json\n{\n  'is_food': 0, # int - 0 or 1 based on whether food/drinks are present (0 = no foods visible, 1 = foods visible)\n  'image_title': '', # str - short food-related title for what foods/drinks are visible in the image, leave blank if no foods present\n  'food_items': [], # list[str] - list of visible edible food item nouns\n  'drink_items': [] # list[str] - list of visible edible drink item nouns\n}\n```\n"}]}]

@spaces.GPU
def extract_foods_from_image(input_image):
    input_image = input_image.resize(size=(512, 512))
    input_message = create_message(input_image=input_image)

    # Get outputs from base model (not fine-tuned)
    original_pipeline_output = original_pipeline(text=[input_message],
                                                 max_new_tokens=OUTPUT_TOKENS)

    outputs_pretrained = original_pipeline_output[0][0]["generated_text"][-1]["content"]

    # Get outputs from fine-tuned model (fine-tuned on food images)
    ft_pipe_output = ft_pipe(text=[input_message],
                             max_new_tokens=OUTPUT_TOKENS)
    outputs_fine_tuned = ft_pipe_output[0][0]["generated_text"][-1]["content"]

    return outputs_pretrained, outputs_fine_tuned

demo_title = "🥑➡️📝 FoodExtract-Vision with a fine-tuned SmolVLM2-500M"
demo_description = """* **Base model:** https://huggingface.co/HuggingFaceTB/SmolVLM2-500M-Video-Instruct
* **Fine-tuning dataset:** https://huggingface.co/datasets/mrdbourke/FoodExtract-1k-Vision (1k food images and 500 not food images)
* **Fine-tuned model:** https://huggingface.co/aegean-ai/FoodExtract-Vision-SmolVLM2-500M-fine-tune-v1
* **Adapted from:** Daniel Bourke's tutorial, https://github.com/mrdbourke/learn-huggingface

## Overview

Extract food and drink items in a structured way from images.

The original model outputs fail to capture the desired structure. But the fine-tuned model sticks to the output structure quite well.

However, the fine-tuned model could definitely be improved with respects to its ability to extract the right food/drink items.

Both models use the input prompt:

````
Classify the given input image into food or not and if edible food or drink items are present, extract those to a list. If no food/drink items are visible, return empty lists.

Only return valid JSON in the following form:

```json
{
  'is_food': 0, # int - 0 or 1 based on whether food/drinks are present (0 = no foods visible, 1 = foods visible)
  'image_title': '', # str - short food-related title for what foods/drinks are visible in the image, leave blank if no foods present
  'food_items': [], # list[str] - list of visible edible food item nouns
  'drink_items': [] # list[str] - list of visible edible drink item nouns
}
```
````

Except one model has been fine-tuned on the structured data whereas the other hasn't.

Notable next steps would be:
* **Remove the input prompt:** Just train the model to go straight from image -> text (no text prompt on input), this would save on inference tokens.
* **Fine-tune on more real-world data:** Right now the model is only trained on 1k food images (from Food101) and 500 not food (random internet images), training on real world data would likely significantly improve performance.
* **Fix the repetitive generation:** The model can sometimes get stuck in a repetitive generation pattern, e.g. "onions", "onions", "onions", etc. We could look into patterns to help reduce this.
"""

demo = gr.Interface(
    fn=extract_foods_from_image,
    inputs=gr.Image(type="pil"),
    title=demo_title,
    description=demo_description,
    outputs=[gr.Textbox(lines=4, label="Original Model (not fine-tuned)"),
             gr.Textbox(lines=4, label="Fine-tuned Model")],
    examples=[["examples/example_1.jpeg"],
              ["examples/example_2.jpeg"],
              ["examples/example_3.jpeg"]],
)

if __name__ == "__main__":
    demo.launch(share=False)

In [ ]:
%%writefile demos/FoodExtract-Vision-v1/README.md
---
title: FoodExtract-Vision Fine-tuned VLM Structured Data Extractor
emoji: 🍟➡️📝
colorFrom: green
colorTo: blue
sdk: gradio
app_file: app.py
pinned: false
license: apache-2.0
---

Fine-tuned SmolVLM2-500M to extract food and drink items from images.

Input can be any kind of image and output will be a formatted string such as the following:

```json
{'is_food': 0, 'image_title': '', 'food_items': [], 'drink_items': []}
```

Or for an image of food:

```json
{'is_food': 1, 'image_title': 'fried calamari', 'food_items': ['fried calamari'], 'drink_items': []}
```

In [ ]:
%%writefile demos/FoodExtract-Vision-v1/requirements.txt
num2words
spaces
transformers
torch
accelerate
gradio
torchvision

In [ ]:
# Save three validation images as the demo's examples
for i, sample in enumerate(val_dataset[:3], start=1):
    image = sample["messages"][1]["content"][0]["image"].convert("RGB")
    image.save(f"demos/FoodExtract-Vision-v1/examples/example_{i}.jpeg")
print(sorted(os.listdir("demos/FoodExtract-Vision-v1/examples")))


## Upload the demo to the Hugging Face Hub

This step also runs only when `PUSH_TO_HUB` is `True`. It creates a public Space that runs the demo.

In [ ]:
from huggingface_hub import HfApi

if PUSH_TO_HUB:
    api = HfApi()
    account = HF_ACCOUNT or api.whoami()["name"]
    space_id = f"{account}/FoodExtract-Vision-v1"

    # Point the demo at the model this run uploaded, not the one app.py was written with
    model_id = f"{account}/FoodExtract-Vision-SmolVLM2-500M-fine-tune-v1"
    app_path = "demos/FoodExtract-Vision-v1/app.py"
    with open(app_path) as f:
        app_source = f.read()
    with open(app_path, "w") as f:
        f.write(app_source.replace("aegean-ai/FoodExtract-Vision-SmolVLM2-500M-fine-tune-v1", model_id))

    # Create a public Gradio Space and upload the demo folder to it
    api.create_repo(space_id, repo_type="space", space_sdk="gradio", private=False, exist_ok=True)
    api.upload_folder(
        repo_id=space_id,
        folder_path="demos/FoodExtract-Vision-v1/",
        path_in_repo=".",
        repo_type="space",
        commit_message="Upload the FoodExtract-Vision demo",
    )
    print(f"[INFO] Uploaded the demo to https://huggingface.co/spaces/{space_id}")
else:
    print("[INFO] PUSH_TO_HUB is False, so the demo was not uploaded")


## Implementation notes

* **Data types:** Use the same data type for the model and its inputs. For example, use quantized inputs with a quantized model and bfloat16 inputs with a bfloat16 model.
* **Prompt inputs:** Without `transformers.pipeline`, you may need to include `system_prompt` in `user_prompt` to avoid errors.

## Further experiments

1. **Formalize evaluation:** Compare the model with a model such as Qwen3-VL-8B.

2. **Improve and scale data sampling:** Add more varied samples when the model makes mistakes. For example, add more than 1,000 real-world food images to Food101.

3. **Remove the input prompt:** The current model is fine-tuned with a text prompt and an image. It could instead map the image directly to JSON and use fewer input tokens.

4. **Reduce repetitive generation with reinforcement learning:** The model sometimes repeats outputs such as "onion", "onion", "onion", "onion". Supervised fine-tuning cannot fix this directly, because it trains the model to copy one reference answer token by token and never shows it the effect of its own mistakes. Reinforcement learning works on the model's own outputs instead. The model generates several answers to the same image, a reward function scores each one, and the update makes the higher-scoring answers more likely. For this task the reward can be computed by a program, without human ratings: whether the answer parses as JSON, whether it has the required fields, how many listed items match the label, and a penalty for repeated items. This is RL with verifiable rewards. [VLM post-training with GRPO](/aiml-common/lectures/reasoning/vlm-reasoning/qwen3-vl-2b-instruct/vlm_grpo_trl) applies it to Qwen3-VL-2B.

5. **Fine-tune on another dataset:** Apply the process to another structured data task, such as extracting details from an invoice or poster. A model may perform better on examples that were more common in its training data.

## Resources

### Videos

The following YouTube livestreams show the unedited coding and problem-solving process.

* [Part 1](https://youtube.com/live/X-nYCCviCyI?feature=share): Creating a VLM dataset
* [Part 2](https://youtube.com/live/Lgcp9hBqWEM?feature=share): Fine-tuning a VLM with LoRA and QLoRA and resolving errors
* [Part 3](https://youtube.com/live/cZVU559BLLM?feature=share): Fine-tuning SmolVLM2 without LoRA or QLoRA, uploading it to the Hugging Face Hub, and publishing a demo. A future video will cover LoRA and QLoRA.

### Hardware

Training uses bfloat16, so it needs an NVIDIA GPU that supports it, such as an L4 or A100 in Google Colab. The run on this page used one NVIDIA RTX PRO 6000 Blackwell GPU.

### Models, data and demos

* Dataset: [FoodExtract-1k-Vision](https://huggingface.co/datasets/mrdbourke/FoodExtract-1k-Vision)
* Base model: [SmolVLM2-500M-Video-Instruct](https://huggingface.co/HuggingFaceTB/SmolVLM2-500M-Video-Instruct)
* Demo from the tutorial: [mrdbourke/FoodExtract-Vision-v1](https://huggingface.co/spaces/mrdbourke/FoodExtract-Vision-v1)
* Fine-tuned model from the tutorial: [mrdbourke/FoodExtract-Vision-SmolVLM2-500M-fine-tune-v1](https://huggingface.co/mrdbourke/FoodExtract-Vision-SmolVLM2-500M-fine-tune-v1)
* Fine-tuned model from the run on this page: [aegean-ai/FoodExtract-Vision-SmolVLM2-500M-fine-tune-v1](https://huggingface.co/aegean-ai/FoodExtract-Vision-SmolVLM2-500M-fine-tune-v1)
* Demo from the run on this page: [aegean-ai/FoodExtract-Vision-v1](https://huggingface.co/spaces/aegean-ai/FoodExtract-Vision-v1)

### Documentation and extras
* Dataset format for VLM fine-tuning: [Vision Datasets](https://huggingface.co/docs/trl/en/dataset_formats#vision-datasets)
* Hugging Face blog post of what is a VLM: [VLMs 2025](https://huggingface.co/blog/vlms-2025)
* Hugging Face docs of fine-tuning SmolVLM: [Smol_VLM_FT.ipynb](https://github.com/huggingface/smollm/blob/main/vision/finetuning/Smol_VLM_FT.ipynb)
* Google Docs example of fine-tuning a VLM: [Hugging Face Vision Fine-tune QLoRA](https://ai.google.dev/gemma/docs/core/huggingface_vision_finetune_qlora)
* Another doc example of inference with a VLM: [Gemma 3n Multimodal Understanding with HF](https://github.com/google-gemini/gemma-cookbook/blob/main/Gemma/%5BGemma_3n%5DMultimodal_understanding_with_HF.ipynb)
* LoRA without regret blog post: [LoRA Without Regret](https://huggingface.co/docs/trl/main/en/lora_without_regret)
    * Thinking Machines blog: [LoRA](https://thinkingmachines.ai/blog/lora/)
* Example of LoRA in the wild: [Apple Intelligence Foundation Models Adapter](https://developer.apple.com/apple-intelligence/foundation-models-adapter/)
* PEFT (Parameter Efficient Fine-tuning) GitHub - [huggingface/peft](https://github.com/huggingface/peft)
* SmolVLM Docling paper - [arXiv:2503.11576](https://arxiv.org/pdf/2503.11576)
    * Notes:
        * LLM alignment: Freeze the vision encoder and train the rest of the network on the new output format.
        * Unfrozen vision encoder: After LLM alignment, unfreeze the vision encoder. Train the model on the pretraining datasets from section 4.1 and the task-specific conversion datasets for tables, code, equations, and charts.
        * Workflow:
            * Stage 1 (frozen vision encoder): Freeze the vision encoder and train the LLM components to produce the target structure, such as JSON for food and drink.
                * Create an `SFTTrainer` instance with its own optimizer and a frozen vision backbone.
            * Stage 2 (unfrozen full model): Unfreeze the vision encoder and train the full model on the target structure.
                * Create a second trainer instance, such as `Trainer_2`, with a new optimizer and make the full model trainable.

## Definitions

* Base Model: A pretrained model that you can download from a source such as Hugging Face and fine-tune for your task.
* Small Vision Language Model: A VLM with fewer than 1B (1 billion) parameters. A model of this size can be deployed to mobile and edge devices.
    * Models larger than 1B parameters can also run on edge devices. Here, *small* means fewer than 1B parameters.
* PEFT: Parameter Efficient Fine-Tuning trains a subset of the model parameters.
* LoRA: Low Rank Adaptation, a form of PEFT. https://arxiv.org/abs/2106.09685
    * You fine-tune an adapter for the original model and can then apply that adapter to the model.
    * You can train multiple adapters and select one for each use case.
* QLoRA: Quantized LoRA fine-tunes an adapter.
    * It uses quantized weights for more parameter-efficient fine-tuning than LoRA.

> This section does not use PEFT. The model is small enough to fine-tune its whole language model on one GPU. PEFT pays off for larger models, where full fine-tuning needs more memory than one GPU has.

## PyTorch reference

| PyTorch class | Description |
|---|---|
| [`nn.Conv2d`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Conv2d.html) | Applies a 2D convolution over an input signal composed of several input planes. |
| [`nn.MultiheadAttention`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.MultiheadAttention.html) | Allows the model to jointly attend to information from different representation subspaces. |
| [`nn.LayerNorm`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.LayerNorm.html) | Applies Layer Normalization over a mini-batch of inputs. |
| [`nn.Linear`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Linear.html) | Applies an affine linear transformation to the incoming data: $y = xA^T + b$. |
| [`nn.Embedding`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Embedding.html) | A simple lookup table that stores embeddings of a fixed dictionary and size. |
| [`nn.GELU`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.GELU.html) | Applies the Gaussian Error Linear Units function. |